In [1]:
import torch
from torch import nn
from torch.nn import functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Current device: {device}")

Current device: cuda


In [ ]:
# hyper-parameters
num_epochs, lr = 2000, 1
num_hiddens = 256
batch_size, num_steps = 64, 35


In [2]:
import matplotlib.pyplot as plt
from IPython import display


class AnimatorNoTest:
    """单图实时刷新：适合 RNN 困惑度等只有 train 曲线的场景。"""

    def __init__(self, xlabel='epoch', ylabel=None, xlim=None, ylim=None,
                 legend=None,
                 fmts=('-', 'm--', 'g-.', 'r:', 'c-', 'y--', 'k-.', 'b:'),
                 figsize=(5, 3)):
        self.xlabel = xlabel
        self.ylabel = ylabel
        self.xlim = xlim
        self.ylim = ylim
        self.legend = list(legend) if legend else []
        self.fmts = list(fmts)
        self.X, self.Y = None, None
        self.fig, self.ax = plt.subplots(figsize=figsize)

    @staticmethod
    def _as_lines(values):
        if values is None:
            return []
        if isinstance(values, (list, tuple)):
            return list(values)
        return [values]

    def _ensure_lines(self, n):
        if self.X is None:
            self.X = [[] for _ in range(n)]
            self.Y = [[] for _ in range(n)]
        elif n > len(self.X):
            self.X.extend([] for _ in range(n - len(self.X)))
            self.Y.extend([] for _ in range(n - len(self.Y)))

    def add(self, x, y):
        """追加点并重绘。y 可为标量或与 legend 等长的序列。"""
        y = self._as_lines(y)
        x = self._as_lines(x)
        if len(x) == 1:
            x = x * len(y)
        elif len(x) != len(y):
            raise ValueError(
                f'x 与 y 长度不一致: len(x)={len(x)}, len(y)={len(y)}'
            )
        self._ensure_lines(len(y))
        for i, (xi, yi) in enumerate(zip(x, y)):
            if xi is not None and yi is not None:
                self.X[i].append(xi)
                self.Y[i].append(yi)

        self.ax.cla()
        for i, (xs, ys) in enumerate(zip(self.X, self.Y)):
            fmt = self.fmts[i % len(self.fmts)]
            label = self.legend[i] if i < len(self.legend) else None
            self.ax.plot(xs, ys, fmt, label=label)
        if self.xlim:
            self.ax.set_xlim(self.xlim)
        if self.ylim:
            self.ax.set_ylim(self.ylim)
        if self.xlabel:
            self.ax.set_xlabel(self.xlabel)
        if self.ylabel:
            self.ax.set_ylabel(self.ylabel)
        if self.legend:
            self.ax.legend(self.legend)
        self.ax.grid(True)
        self.fig.tight_layout()
        display.clear_output(wait=True)
        display.display(self.fig)

    def close(self):
        plt.close(self.fig)

In [3]:
import os
import re
import hashlib
from urllib import request
path = '/home/dddsx259/Study/MyStudyNote/Machine Learning & Deep Learning/d2l-zh/practice/data/斗破苍穹.txt'

def read_doupo():
    with open(path, 'r') as f:
        lines = f.readlines()
    return lines

In [4]:
def tokenize_chinese(lines):
    """将文本行拆分为单词或字符词元。"""
    return [list(line) for line in lines]

In [5]:
import collections

def count_corpus(tokens):
    if isinstance(tokens, list):
        tokens = [tk for line in tokens for tk in line]
    return collections.Counter(tokens)

In [6]:
class Vocab:
    """文本词表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        # 按出现频率排序
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                   reverse=True)
        # 未知词元的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx
                             for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):  # 未知词元的索引为0
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

In [7]:
import random

class SeqDataLoader:
    def __init__(self, batch_size, num_steps, corpus=None, vocab=None, mode='random'):
        assert not (corpus is None or vocab is None)
        self.corpus = corpus
        self.vocab = vocab
        if mode == 'random':
            self.data_iter_fn = self.seq_data_iter_random
        else:
            self.data_iter_fn = self.seq_data_iter_sequential
        self.batch_size = batch_size
        self.num_steps = num_steps
    
    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)
    
    def seq_data_iter_random(self, corpus, batch_size, num_steps):
        corpus = corpus[random.randint(0, num_steps - 1):]
        num_subseqs = (len(corpus) - 1) // num_steps
        initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
        random.shuffle(initial_indices)
        def data(pos):
            # 返回从pos位置开始的长度为num_steps的序列
            return corpus[pos: pos + num_steps]

        num_batches = num_subseqs // batch_size
        for i in range(0, batch_size * num_batches, batch_size):
            # 在这里，initial_indices包含子序列的随机起始索引
            initial_indices_per_batch = initial_indices[i: i + batch_size]
            X = [data(j) for j in initial_indices_per_batch]
            Y = [data(j + 1) for j in initial_indices_per_batch]
            yield torch.tensor(X), torch.tensor(Y)
    
    def seq_data_iter_sequential(self, corpus, batch_size, num_steps):
        offset = random.randint(0, num_steps)
        num_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size
        Xs = torch.tensor(corpus[offset: offset + num_tokens])
        Ys = torch.tensor(corpus[offset + 1: offset + 1 + num_tokens])
        Xs, Ys = Xs.reshape(batch_size, -1), Ys.reshape(batch_size, -1)
        num_batches = Xs.shape[1] // num_steps
        for i in range(0, num_steps * num_batches, num_steps):
            X = Xs[:, i: i + num_steps]
            Y = Ys[:, i: i + num_steps]
            yield X, Y
        

In [8]:
def normal(shape, device):
    return torch.randn(size=shape, device=device) * 0.01

In [9]:
class Accumulator:
    def __init__(self, num):
        self.data = [0.0] * num
    
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
    
    def reset(self):
        self.data = [0.0] * len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [10]:
def three(num_inputs, num_hiddens, device):
    return (normal((num_inputs, num_hiddens), device), 
            normal((num_hiddens, num_hiddens), device), 
            torch.zeros(num_hiddens, device=device))

In [11]:
lines = read_doupo()[:1145+67]
lines, prefix = lines[:1145], lines[1145:]
tokens = tokenize_chinese(lines)
for i in range(5):
    print(lines[i*3])
    print(tokens[i*3])
    print()

斗破苍穹

['\ufeff', '斗', '破', '苍', '穹', '\n']



['\n']



['\n']

第一章 陨落的天才

['第', '一', '章', ' ', '陨', '落', '的', '天', '才', '\n']



['\n']



In [12]:
vocab = Vocab(tokens, min_freq=2)
corpus = [vocab[token] for line in tokens for token in line]
print(f"Lenth of corpus: {len(corpus)}, lenth of vocab: {len(vocab)}")

Lenth of corpus: 36028, lenth of vocab: 1388


In [ ]:
train_iter = SeqDataLoader(batch_size, num_steps, corpus=corpus, vocab=vocab, mode='random')


# 3. Initialization of RNN model

In [14]:
def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    
    W_xz, W_hz, b_z = three(num_inputs, num_hiddens, device)
    W_xr, W_hr, b_r = three(num_inputs, num_hiddens, device)
    W_xh, W_hh, b_h = three(num_inputs, num_hiddens, device)
    W_hq = normal((num_hiddens, num_outputs), device)
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

def init_rnn_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),)
    

In [15]:
def gru(inputs, state, params):
    W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        Z = torch.sigmoid(X @ W_xz + H @ W_hz + b_z)
        R = torch.sigmoid(X @ W_xr + H @ W_hr + b_r)
        H_tilda = torch.tanh(X @ W_xh + R * (H @ W_hh + b_h))
        H = Z * H + (1 - Z) * H_tilda
        Y = H @ W_hq + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

In [16]:
class RNNModelScratch:
    def __init__(self, vocab_size, num_hiddens, device, get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens, self.init_state, self.forward_fn = vocab_size, num_hiddens, init_state, forward_fn
        self.params = get_params(vocab_size, num_hiddens, device)
    
    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)
    
    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)

In [17]:
def sgd(params, lr, batch_size):
    for param in params:
        param.data -= lr * param.grad / batch_size
        param.grad.zero_()

In [18]:
def grad_clipping(net, theta):
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum((p.grad**2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta/norm
    

In [19]:
def predict_rnn(prefix, num_preds, net, vocab, device):
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]:
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds):
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return ''.join([vocab.idx_to_token[i] for i in outputs])

In [ ]:
net = RNNModelScratch(len(vocab), num_hiddens, device, get_params, init_rnn_state, gru)


# 4. Training and prediction.

In [21]:
#try predict a simple sentence
prefix = tokenize_chinese(prefix)
prefix = [token for line in prefix for token in line]
print(prefix)
predict_rnn(prefix[0], 20, net, vocab, device)

['\n', ' ', ' ', ' ', ' ', '萧', '炎', '手', '中', '的', '手', '链', '，', '从', '材', '质', '上', '看', '，', '明', '显', '只', '是', '一', '个', '不', '会', '超', '过', '五', '枚', '金', '币', '的', '地', '摊', '货', '，', '而', '他', '的', '木', '灵', '之', '链', '，', '却', '是', '正', '宗', '的', '魔', '晶', '首', '饰', '，', '在', '购', '买', '之', '时', '，', '足', '足', '花', '费', '了', '一', '千', '多', '枚', '金', '币', '，', '两', '条', '手', '链', '，', '不', '论', '式', '样', '，', '价', '格', '，', '以', '及', '实', '用', '程', '度', '，', '都', '是', '天', '差', '地', '别', '，', '毫', '无', '半', '点', '比', '较', '性', '，', '所', '以', '，', '加', '列', '奥', '看', '着', '萧', '炎', '竟', '然', '给', '薰', '儿', '这', '位', '美', '少', '女', '如', '此', '寒', '碜', '的', '首', '饰', '，', '实', '在', '是', '有', '些', '忍', '不', '住', '的', '出', '言', '讥', '诮', '道', '：', '“', '萧', '炎', '少', '爷', '，', '虽', '然', '早', '知', '道', '你', '在', '自', '己', '家', '族', '中', '地', '位', '不', '高', '，', '可', '…', '可', '你', '也', '不', '用', '如', '此', '寒', '碜', '薰', '儿', '小', '姐', '吧', '?', '”', '\n', '\n', ' ', ' ', ' ', ' ', '

'\n岁嘶资两力睛跪脚次钱发孔口速墨鲜抚紫冲涨'

In [ ]:
import math

def train_epoch_rnn(net, train_iter, loss, updater, device, use_random_iter):
    state = None
    metric = Accumulator(2)
    for X, y in train_iter:
        if state is None or use_random_iter:
            state = net.begin_state(X.shape[0], device)
        else:
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                state.detach_()
            else:
                for s in state:
                    s.detach_()
        y = y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long()).mean()
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        metric.add(l.detach() * y.numel(), y.numel())
    return math.exp(metric[0]/metric[1])

def train_rnn(net, train_iter, vocab, lr, num_epochs, device, use_random_iter=False):
    last_perplexity = float('inf')
    count = 0
    loss = nn.CrossEntropyLoss()
    animator = AnimatorNoTest(
        xlabel='epoch', ylabel='perplexity', legend=['train'], xlim=[10, num_epochs])
    if isinstance(net, nn.Module):
        net = net.to(device)
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_rnn(prefix, 50, net, vocab, device)
    print_every = max(1, num_epochs // 100)
    for epoch in range(num_epochs):
        perplexity = train_epoch_rnn(net, train_iter, loss, updater, device, use_random_iter)
        if (epoch + 1) % print_every == 0:
            print(f'epoch {epoch + 1}, perplexity {perplexity:.1f}')
            animator.add(epoch + 1, [perplexity])
        if last_perplexity - perplexity < 1e-3 and last_perplexity - perplexity > 0:
            count += 1
            if count >= 5:
                print(f'早停于 epoch {epoch + 1}: 困惑度连续5次下降小于1e-3, 当前 {perplexity:.1f}')
                break
        else:
            count = 0
        last_perplexity = perplexity
    print(f'困惑度: {perplexity:.1f}')
    print(predict(prefix))
train_rnn(net, train_iter, vocab, lr, num_epochs, device)


In [23]:
predict_rnn(prefix, 100, net, vocab, device)

'\n    萧炎手中的手链，从材质上看，明显只是一个不会超过五枚金币的地摊货，而他的木灵之链，却是正宗的魔晶首饰，在购买之时，足足花<unk>了一千多枚金币，两条手链，不论式样，价格，以及实用程度，都是天差地别，毫无半点比较性，所以，加列奥看着萧炎竟然给薰儿这位美少女如此寒碜的首饰，实在是有些忍不住的出言讥<unk>道：“萧炎少爷，虽然早知道你在自己家族中地位不高，可…可你也不用如此寒碜薰儿小姐吧?”\n\n    没有理会加列奥的讥笑，萧炎对盯着手链忽然有些发<unk>的少女扬了扬手，有些不耐的道：“到底要不要啊?不要就丢了，反正就买成两三个金币而已。”\n\n    “<unk>…”听着萧炎的话，不仅加列奥失笑，就是连其身旁的一群属下，也是嘲讽的<unk>笑了起来。\n\n    然而嘲讽的<unk>笑并未持续多久，便是犹如被忽然<unk>断了<unk>子一般，<unk>然的断在了<unk><unk>处，众人张着嘴巴惊愕的模样极为滑<unk>。\n\n    原本有些发愣的少女被萧炎的举动惊<unk>了过来，双手几乎是在潜意识的支配下，<unk>速抢过了手链，手链到手之后，薰儿这才回过神来，自己表现的似乎有点太急切了…\n\n    白皙的精致小脸上浮现一抹淡淡的<unk>红，不过薰儿也非常人，在略微<unk>涩之后，便是落落大方的将将手链套在了光<unk>白皙的皓腕之上，抬头对着萧炎露出清雅的娇笑：“谢谢萧炎哥哥。”\n\n    脸色有些不自然的望着那在萧炎面前露出正常少女姿态的薰儿，加列奥脸庞上有些嫉妒，干笑道：“呵呵，没想到薰儿小姐爱好如此与众不同，我倒是有点失算了。”\n\n    萧炎瞟了一眼面前的加列奥，目光在其胸口处的一枚金星上扫过，心头不由有些诧异：“去年见到这家伙，他应该是九段斗之气吧？没想到今年竟然成功凝聚斗之气旋了，不过二十一岁才成为一名一星斗者，这天赋，勉强只能算作上等吧…”\n\n    见到这家伙还没有离开的打算，萧炎撇了撇嘴，懒懒的话语，并未因为对方的实力而客气几分，萧家与加列家族关系本来就不好，所以他也没必要展现什么低姿态，摸了摸鼻子，萧炎淡淡的道：“加列奥少爷，你的风流习性，整个乌坦城都知道，不过薰儿还小，没空和你玩早恋的游戏，所以，以后麻烦你还是去<unk>害别家<unk>女吧。”\n\n    “以后离他远点。”\n\n